In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q3_data.csv')
df_goldenF = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df_goldenF.head(50)

In [ ]:
# Task 3: Write your code here:
df_goldenF.info()

In [ ]:
# Task 4: Write your code here:
df_goldenF.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df_goldenF):
  missing_values = df_goldenF.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_goldenF)

In [ ]:
cols = df_goldenF.select_dtypes(include=["int"]).columns
df_goldenF[cols].fillna(df_goldenF[cols].mean())
df_goldenF[cols].drop(columns=[cols])

In [ ]:
# Task 2: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(df_goldenF):
  duplicates = df_goldenF.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_goldenF.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df_goldenF)

In [ ]:
# Task 3: Write your code here:

# the data is already encoded

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_for_scale = df_goldenF.drop(columns=['Target'])
data_standard_scaled = standard_scaler.fit_transform(data_for_scale) # Apply fit_transform


In [ ]:
# Task 5: Write your code here:
# Is the target imbalanced?
import seaborn as sns
def check_target_imbalance(df_goldenF, target_column):
  print("Target Distribution:")
  print(df_goldenF['Target'].value_counts(normalize=True))
  sns.countplot(x=df_goldenF['Target'])
  plt.title("Target Distribution")
  plt.show()
check_target_imbalance(df_goldenF, "status")

In [ ]:
# Task 1: Write your code here:
X = df_goldenF.drop("Target", axis=1).astype(float)
y = df_goldenF['Target'].astype(float)

In [ ]:
pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

catBoost = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4 )


# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)



        # Train and predict
    catBoost.fit(X_train, y_train)
    y_pred = catBoost.predict(X_test)

    f1Score= f1_score(y_test, y_pred)
    print(f1Score)

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = catBoost['LASSO Regression'].coef_
coeffs['Ridge'] = catBoost['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (catBoost, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{catBoost} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
print('the most important feature: p_2')

In [ ]:
# Task Bonus: Write your code here:
